# Intro

⚠️ Training a tokenizer is not the same as training a model!
- Model training uses stochastic gradient descent to make the loss a little bit smaller for each batch.
    - It’s randomized by nature (meaning you have to set some seeds to get the same results when doing the same training twice).
- Training a tokenizer is a statistical process that tries to identify which subwords are the best to pick for a given corpus, and the exact rules used to pick them depend on the tokenization algorithm.
    - It’s deterministic, meaning you always get the same results when training with the same algorithm on the same corpus.

# Assembling a corpus

- Let’s say we want to train **GPT-2** from scratch, but in a language __other than English__.
- Our first task will be to gather lots of data in that language in a training corpus.
- We will use a specialized English language: **Python code**.
- The [🤗 Datasets](https://github.com/huggingface/datasets) library can help us assemble a corpus of Python source code.
- We’ll use the usual `load_dataset()` function to download and cache the [CodeSearchNet](https://github.com/github/CodeSearchNet) dataset.
    - it contains millions of functions from open source libraries on GitHub in several programming languages.

## Download the raw data

In [ ]:
from datasets import load_dataset
raw_datasets = load_dataset("code_search_net", "python")

In [ ]:
# print(raw_datasets.keys())
# print("=====================")
# print(raw_datasets['train'][123456].keys())
# print("=====================")
# print(raw_datasets['train'][123456]['language'])
# print("=====================")
# print(raw_datasets['train'][123456]['func_documentation_string'])
# print("=====================")
# print(raw_datasets['train'][123456]['func_documentation_tokens'])
# print("=====================")
# print(raw_datasets['train'][123456]['whole_func_string'])
# print("=====================")
# print(raw_datasets['train'][123456]['func_code_string'])
# print("=====================")
# print(raw_datasets['train'][123456]['func_code_tokens'])

## Chunkify the data

- Option 1 would create a list of lists of 1,000 texts each, but would load everything in memory.
- Options 2 creates a python **generator**.
    - The texts will only be loaded batch by batch into memory when you need them.
    - Only 1,000 texts at a time will be loaded
    - The important fact with a **generator** object is that it **can only be used once**.
        - That’s why we define a function that returns a generator

In [ ]:
bsz = 1000

# # Option 1
# # Don't uncomment the following line unless your dataset is small!
# training_corpus = [
#     raw_datasets["train"][i: i + bsz]["whole_func_string"]
#     for i in range(0, len(raw_datasets["train"]), bsz)
# ]
# # Option 2
def get_training_corpus(bsz):
    return (
        raw_datasets["train"][i : i + bsz]["whole_func_string"]
        for i in range(0, len(raw_datasets["train"]), bsz)
    )

training_corpus = get_training_corpus(bsz)
training_corpus

# Training a new tokenizer

- First, we need to load the tokenizer we want to pair with our model (here, **GPT-2**)
- It’s a good idea to do this to avoid starting entirely from scratch.
- This way, we won’t have to specify anything about the **tokenization algorithm** or the **special tokens** we want to use
- our new tokenizer will be exactly the same as GPT-2, and **the only thing that will change is the vocabulary**, which will be determined by the training on our corpus.

In [ ]:
from transformers import AutoTokenizer
old_tokenizer = AutoTokenizer.from_pretrained("gpt2")

First let’s have a look at how this tokenizer would treat an example function:

In [ ]:
example = \
'''
def add_numbers(a, b):
    """Add the two numbers `a` and `b`."""
    return a + b
'''

old_tokens = old_tokenizer.tokenize(example)
print(old_tokens)

- This tokenizer has a few special symbols, like `Ġ` and `Ċ`, which denote spaces and newlines, respectively.
    - As we can see, this is not too efficient
    - the tokenizer returns individual tokens for each spacewhen it could group together indentation levels
        - since having sets of four or eight spaces is going to be very common in code.
- It also split the function name a bit weirdly, not being used to seeing words with the `_` character.

Let’s train a new tokenizer and see if it solves those issues. For this, we’ll use the method `train_new_from_iterator()`
- Note that `AutoTokenizer.train_new_from_iterator()` only works if the tokenizer you are using is a “fast” tokenizer.
- The `AutoTokenizer` API always selects the fast tokenizer for you if it’s available.

In [ ]:
# !ls -l ~/.cache/huggingface/datasets/code_search_net/python/0.0.0/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555
print(f"Dataset size: {raw_datasets['train'].info.dataset_size / (1024**3):.4f} GB")
tokenizer = old_tokenizer.train_new_from_iterator(
    text_iterator = training_corpus, vocab_size = 52000
)

In [ ]:
new_tokens = tokenizer.tokenize(example)
print(new_tokens)
print("================")
print(f"old token count: {len(old_tokens)}")
print(f"new token count: {len(new_tokens)}")

- Here we again see the special symbols `Ġ` and `Ċ` that denote spaces and newlines
- We can also see that our tokenizer learned some tokens that are highly specific to a corpus of Python functions
    - for example, there is a `ĊĠĠĠ` token that represents an indentation,
    - and a `Ġ"""` token that represents the three quotes that start a docstring.
- The tokenizer also correctly split the function name on `_`.
- This is quite a compact representation; comparatively, yileding a smaller set of tokens for the same example.

In [ ]:
example = \
"""
class LinearLayer():
    def __init__(self, input_size, output_size):
        self.weight = torch.randn(input_size, output_size)
        self.bias = torch.zeros(output_size)

    def __call__(self, x):
        return x @ self.weights + self.bias
"""
print(tokenizer.tokenize(example))

- In addition to the token corresponding to an indentation, here we can also see a token for a double indentation: `ĊĠĠĠĠĠĠĠ`.
- The special Python words like `class`, `init`, `call`, `self`, and `return` are each tokenized as one token,
- And we can see that as well as splitting on `_` and `.` the tokenizer correctly splits even camel-cased names: `LinearLayer` is tokenized as `["ĠLinear", "Layer"]`.

# Saving the tokenizer

## HF Login

In [ ]:
from huggingface_hub import login

# # Colab ========================
# from google.colab import userdata
# HF_TOKEN = userdata.get('HF_TOKEN')

# Local ========================
import os
from dotenv import load_dotenv
load_dotenv()
HF_TOKEN = os.getenv('HF_TOKEN')

# Login
if HF_TOKEN:
    login(HF_TOKEN)
    print("Successfully logged in to Hugging Face!")
else:
    print("HF_TOKEN not found in Colab secrets.")

## Push to HF hub

In [ ]:
tokenizer.push_to_hub("code-search-net-tokenizer")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("amin-oj/code-search-net-tokenizer")